# Critical Input DEQN: Natural Benchmark

This notebook trains and saves the auxiliary flexible-price benchmark network. Policy notebooks reuse this checkpoint instead of retraining the benchmark.

In [ ]:
# Configure paths and natural-benchmark training settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'
OUT = ARTIFACT_ROOT / 'natural'
OUT.mkdir(parents=True, exist_ok=True)

NATURAL_STEPS = 50_000
QMC_TRAIN = 512
QMC_VAL = 1024
N_VAL_STATES = 2048
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
LOG_EVERY = 100
TARGET_RMS = None
TARGET_MAX_ABS = None
EARLY_STOP_PATIENCE = None
MIN_STEPS_BEFORE_STOP = None
STOP_VAL_STATES = 512
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'
print(ROOT)
print(OUT)

# Stream subprocess output line by line in Colab instead of waiting silently.
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)



In [ ]:
# Train the auxiliary flexible-price benchmark network.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_train',
    '--output-dir', str(OUT),
    '--policies', '',
    '--natural-steps', str(NATURAL_STEPS),
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
]
if TARGET_RMS is not None:
    cmd += ['--target-rms', str(TARGET_RMS)]
if TARGET_MAX_ABS is not None:
    cmd += ['--target-max-abs', str(TARGET_MAX_ABS)]
if EARLY_STOP_PATIENCE is not None:
    cmd += ['--early-stop-patience', str(EARLY_STOP_PATIENCE)]
if MIN_STEPS_BEFORE_STOP is not None:
    cmd += ['--min-steps-before-stop', str(MIN_STEPS_BEFORE_STOP)]
run_stream(cmd, cwd=ROOT)

In [ ]:
# Inspect out-of-sample residual diagnostics for the benchmark.
with (OUT / 'natural_eval.json').open('r', encoding='utf-8') as fh:
    natural_eval = json.load(fh)
natural_eval